In [ ]:
!pip install -q --upgrade pip
!pip install -q --upgrade --force-reinstall "huggingface_hub>=0.34.0"
!pip install -q --upgrade "transformers>=4.51.0" accelerate datasets peft bitsandbytes trl sentencepiece protobuf
!pip install -q --upgrade --no-cache-dir "unsloth_zoo>=2026.3.4"
!pip install -q --upgrade --no-cache-dir "unsloth @ git+https://github.com/unslothai/unsloth.git"
import importlib.metadata as _im
print("huggingface_hub:", _im.version("huggingface_hub"))
print("transformers:", _im.version("transformers"))
print("unsloth_zoo:", _im.version("unsloth_zoo"))
print("unsloth:", _im.version("unsloth"))
from unsloth import FastLanguageModel
print("Unsloth import OK")
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
    print("HF login OK")
except Exception as e:
    print("Add HF_TOKEN in Kaggle Secrets:", e)

In [ ]:
from unsloth import FastLanguageModel
import torch
BASE_MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Gemma 4 loaded:", BASE_MODEL_NAME)
model.print_trainable_parameters()

In [ ]:
import json
training_data = []
SYSTEM_PROMPT = """You are Sahaj, an AI assistant for Indian citizens.
Your job: understand the user's input and extract structured information.
Always respond with valid JSON containing:
- "intent": one of ["scheme_discovery", "legal_aid", "document_scan", "general_query"]
- relevant extracted fields based on intent
Do NOT suggest specific schemes or laws. Only extract the user's profile/situation."""
scheme_extraction_examples = [
    {
        "input": "Main ek kisan hoon, Rajasthan se, 2 acre zameen hai, ek beti hai 5 saal ki, BPL card hai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "farmer", "state": "Rajasthan", "land_acres": 2, "income_category": "BPL", "gender": "male", "family": [{"relation": "daughter", "age": 5}]}}, ensure_ascii=False)
    },
    {
        "input": "Hum chhote kisan hain MP se, 1 acre mein gehu ugaate hain, umr 52, SC category",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "farmer", "state": "Madhya Pradesh", "land_acres": 1, "crop": "wheat", "age": 52, "category": "SC", "gender": "male"}}, ensure_ascii=False)
    },
    {
        "input": "I am a farmer from Punjab, 5 acres, grow rice and wheat, 2 sons both in college, age 48",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "farmer", "state": "Punjab", "land_acres": 5, "crops": ["rice", "wheat"], "age": 48, "gender": "male", "family": [{"relation": "son", "status": "college"}, {"relation": "son", "status": "college"}]}}, ensure_ascii=False)
    },
    {
        "input": "Mere paas zameen nahi hai, bhumi heen hoon, Bihar se, mazdoori karta hoon khet pe, age 35",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "landless_laborer", "state": "Bihar", "land_acres": 0, "age": 35, "gender": "male", "work_type": "agricultural_labor"}}, ensure_ascii=False)
    },
    {
        "input": "Main mahila kisan hoon, 3 acre hai mere naam pe, Haryana se, OBC, pati nahi hai ab",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "farmer", "state": "Haryana", "land_acres": 3, "land_ownership": "self", "gender": "female", "category": "OBC", "marital_status": "widow"}}, ensure_ascii=False)
    },
    {
        "input": "Tamil Nadu se hoon, 10 acre coconut farm hai, OBC, age 55, wife and 3 children",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "farmer", "state": "Tamil Nadu", "land_acres": 10, "crop": "coconut", "category": "OBC", "age": 55, "gender": "male", "family": [{"relation": "wife"}, {"relation": "children", "count": 3}]}}, ensure_ascii=False)
    },
    {
        "input": "Main machli pakadne ka kaam karta hoon, Kerala coast pe, 40 saal, boat hai choti si",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "fisherman", "state": "Kerala", "age": 40, "gender": "male", "assets": ["small_boat"]}}, ensure_ascii=False)
    },
    {
        "input": "Main ek vidhwa hoon, UP se, 2 bachche hain, BPL card hai, koi aadmi nahi hai ghar mein",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "homemaker", "state": "Uttar Pradesh", "income_category": "BPL", "gender": "female", "marital_status": "widow", "children": 2}}, ensure_ascii=False)
    },
    {
        "input": "Main pregnant hoon, 7 mahina ho gaya, pehla bachcha, gaon mein rehti hoon Jharkhand mein",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"state": "Jharkhand", "gender": "female", "pregnant": True, "pregnancy_month": 7, "first_child": True, "area": "rural"}}, ensure_ascii=False)
    },
    {
        "input": "Meri maa 62 saal ki hai, akeli rehti hain, Chhattisgarh mein, BPL hai, pension nahi aati",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"age": 62, "state": "Chhattisgarh", "gender": "female", "income_category": "BPL", "marital_status": "single", "pension_status": "none"}}, ensure_ascii=False)
    },
    {
        "input": "I am a single mother, 2 daughters age 6 and 9, working as domestic help in Bangalore, no BPL card but very low income",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "domestic_worker", "state": "Karnataka", "city": "Bangalore", "gender": "female", "marital_status": "single_mother", "income_category": "LIG", "family": [{"relation": "daughter", "age": 6}, {"relation": "daughter", "age": 9}]}}, ensure_ascii=False)
    },
    {
        "input": "Main anganwadi worker hoon, 15 saal se kaam kar rahi hoon, Gujarat, age 38, 1 beta hai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "anganwadi_worker", "state": "Gujarat", "age": 38, "gender": "female", "work_experience_years": 15, "family": [{"relation": "son"}]}}, ensure_ascii=False)
    },
    {
        "input": "I am a street vendor selling vegetables in Mumbai, age 45, no bank account",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "street_vendor", "state": "Maharashtra", "city": "Mumbai", "age": 45, "gender": "male", "has_bank_account": False, "vendor_type": "vegetables"}}, ensure_ascii=False)
    },
    {
        "input": "Construction worker hoon, Delhi mein kaam karta hoon lekin ghar Jharkhand mein hai, migrant hoon, 28 saal",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "construction_worker", "work_state": "Delhi", "home_state": "Jharkhand", "age": 28, "gender": "male", "migrant": True}}, ensure_ascii=False)
    },
    {
        "input": "Auto rickshaw chalata hoon Hyderabad mein, permit hai, age 50, ek ladka 12th pass karna chahta hai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "auto_driver", "state": "Telangana", "city": "Hyderabad", "age": 50, "gender": "male", "has_permit": True, "family": [{"relation": "son", "education": "12th_appearing"}]}}, ensure_ascii=False)
    },
    {
        "input": "Gharelu kaamgaar hoon, 3 gharon mein kaam karti hoon, Pune mein, ration card hai, 35 years",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "domestic_worker", "state": "Maharashtra", "city": "Pune", "age": 35, "gender": "female", "has_ration_card": True}}, ensure_ascii=False)
    },
    {
        "input": "Main tailor hoon, apni dukaan hai choti si, Varanasi mein, 42 age, OBC, 2 bache school mein",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "tailor", "state": "Uttar Pradesh", "city": "Varanasi", "age": 42, "gender": "male", "category": "OBC", "has_own_shop": True, "family": [{"relation": "children", "count": 2, "status": "school"}]}}, ensure_ascii=False)
    },
    {
        "input": "Welder ka kaam karta hoon factory mein, Ludhiana, 30 saal, koi insurance nahi hai, daily 500 milta hai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "welder", "work_type": "factory", "state": "Punjab", "city": "Ludhiana", "age": 30, "gender": "male", "has_insurance": False, "daily_wage": 500}}, ensure_ascii=False)
    },
    {
        "input": "Main 12th pass hoon, engineering karna chahta hoon lekin fees nahi de sakta, Bihar, SC category, papa kisan",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "student", "education": "12th_pass", "goal": "engineering", "state": "Bihar", "category": "SC", "gender": "male", "financial_need": True, "parent_occupation": "farmer"}}, ensure_ascii=False)
    },
    {
        "input": "Meri beti 10th mein hai, achhe marks aate hain, Rajasthan, hum ST hain, scholarship mil sakti hai kya",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"family": [{"relation": "daughter", "education": "10th_studying", "academic_performance": "good"}], "state": "Rajasthan", "category": "ST", "looking_for": "scholarship"}}, ensure_ascii=False)
    },
    {
        "input": "I am doing MBBS 2nd year, family income 3 lakh per year, Gujarat, General category, need scholarship",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "student", "education": "MBBS_2nd_year", "state": "Gujarat", "category": "General", "family_income": 300000, "looking_for": "scholarship"}}, ensure_ascii=False)
    },
    {
        "input": "I am a 68 year old retired government servant, my pension comes but I need to do life certificate, I live in Delhi",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "retired_govt", "state": "Delhi", "age": 68, "gender": "male", "has_pension": True, "need": "life_certificate"}}, ensure_ascii=False)
    },
    {
        "input": "Mere papa 72 saal ke hain, koi pension nahi milti, kisan the ab kaam nahi kar sakte, UP village",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"age": 72, "state": "Uttar Pradesh", "area": "rural", "gender": "male", "occupation": "retired_farmer", "has_pension": False, "work_ability": "unable"}}, ensure_ascii=False)
    },
    {
        "input": "Dadi 80 saal ki hain, BPL hai, Odisha village, koi dekhne wala nahi",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"age": 80, "state": "Odisha", "area": "rural", "gender": "female", "income_category": "BPL", "living_alone": True}}, ensure_ascii=False)
    },
    {
        "input": "Hum SC category se hain, Bihar se, ek chota sa business kholna chahte hain, 25 saal umr hai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "aspiring_entrepreneur", "state": "Bihar", "age": 25, "category": "SC", "gender": "male", "business_status": "planning"}}, ensure_ascii=False)
    },
    {
        "input": "I want to start a beauty parlour, I am a woman from Assam, ST community, age 28, some savings",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "aspiring_entrepreneur", "business_type": "beauty_parlour", "state": "Assam", "category": "ST", "age": 28, "gender": "female", "has_savings": True}}, ensure_ascii=False)
    },
    {
        "input": "Chai ki tapri lagana chahta hoon, Indore mein, abhi delivery boy hoon Zomato pe, 23 saal",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "delivery_worker", "aspiration": "tea_stall_business", "state": "Madhya Pradesh", "city": "Indore", "age": 23, "gender": "male"}}, ensure_ascii=False)
    },
    {
        "input": "Main ek physically disabled hoon, wheelchair pe hoon, Kolkata mein rehta hoon, age 30, graduation complete",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"state": "West Bengal", "city": "Kolkata", "age": 30, "gender": "male", "disability": "physical_locomotor", "mobility": "wheelchair", "education": "graduate"}}, ensure_ascii=False)
    },
    {
        "input": "Mera beta 10 saal ka hai, visually impaired hai, special school mein padh raha hai, Jaipur",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"state": "Rajasthan", "city": "Jaipur", "family": [{"relation": "son", "age": 10, "disability": "visual_impairment", "education": "special_school"}]}}, ensure_ascii=False)
    },
    {
        "input": "kuch scheme batao bhai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {}, "missing_info": ["occupation", "state", "age", "category"], "follow_up_hindi": "Aapka kaam kya hai, kis state se hain, aur aapki umr kitni hai?"}, ensure_ascii=False)
    },
    {
        "input": "mujhe paisa chahiye sarkar se",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {}, "missing_info": ["occupation", "state", "age", "income_category", "category"], "follow_up_hindi": "Aap kya kaam karte hain? Kahan rehte hain? Thoda detail bataiye."}, ensure_ascii=False)
    },
    {
        "input": "UP se hoon bas, kisan hoon",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "farmer", "state": "Uttar Pradesh"}, "missing_info": ["land_acres", "age", "category", "family"], "follow_up_hindi": "Kitni zameen hai? Umr kitni hai? Parivar mein kaun kaun hai?"}, ensure_ascii=False)
    },
    {
        "input": "Main ek retired army jawan hoon, 55 years, Uttarakhand, pension milti hai lekin bachon ki padhai ke liye aur paisa chahiye",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "retired_military", "state": "Uttarakhand", "age": 55, "gender": "male", "has_pension": True, "need": "children_education"}}, ensure_ascii=False)
    },
    {
        "input": "Transgender hoon, Delhi mein rehti hoon, 29 age, koi ID nahi hai, kaam dhundh rahi hoon",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"state": "Delhi", "age": 29, "gender": "transgender", "has_id": False, "occupation": "unemployed", "looking_for": "employment"}}, ensure_ascii=False)
    },
    {
        "input": "Mere husband ka accident ho gaya, ab woh bed pe hain, main ghar chalati hoon silai karke, 2 bachche, Assam",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "tailor_home", "state": "Assam", "gender": "female", "family": [{"relation": "husband", "condition": "bedridden_accident"}, {"relation": "children", "count": 2}], "sole_earner": True}}, ensure_ascii=False)
    },
    {
        "input": "Mera ek chota dhaba hai highway pe, Rajasthan, 10 saal se chala raha hoon, ab loan chahiye expand karne ke liye, General category",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "dhaba_owner", "state": "Rajasthan", "business_years": 10, "gender": "male", "category": "General", "looking_for": "business_loan", "purpose": "expansion"}}, ensure_ascii=False)
    },
    {
        "input": "I am a nurse in a private hospital, Lucknow, 32 years, no PF no ESI, husband left me",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "nurse_private", "state": "Uttar Pradesh", "city": "Lucknow", "age": 32, "gender": "female", "has_pf": False, "has_esi": False, "marital_status": "separated"}}, ensure_ascii=False)
    },
    {
        "input": "Pottery banata hoon, mitti ke bartan, Khurja UP se, kumhar jaati, 45 age, arthritis ho gaya haath mein",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "potter", "craft": "pottery", "state": "Uttar Pradesh", "city": "Khurja", "category": "OBC", "age": 45, "gender": "male", "health_issue": "arthritis_hands"}}, ensure_ascii=False)
    },
    {
        "input": "Main barber hoon, nai samaj se, apna salon kholna chahta hoon, Patna, age 27",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "barber", "state": "Bihar", "city": "Patna", "age": 27, "gender": "male", "aspiration": "own_salon"}}, ensure_ascii=False)
    },
    {
        "input": "Rickshaw chalata hoon Kolkata mein, e-rickshaw lena chahta hoon, age 40, koi savings nahi",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "rickshaw_puller", "state": "West Bengal", "city": "Kolkata", "age": 40, "gender": "male", "aspiration": "e_rickshaw", "has_savings": False}}, ensure_ascii=False)
    },
    {
        "input": "Ganga kinare naukari karta hoon, boatman hoon, Varanasi, tourist ko ghaat dikhata hoon, 38 years, OBC",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "boatman", "state": "Uttar Pradesh", "city": "Varanasi", "age": 38, "gender": "male", "category": "OBC", "work_type": "tourism"}}, ensure_ascii=False)
    },
    {
        "input": "Main safai karamchari hoon, nagar nigam mein contractual, Bhopal, 10 saal se, SC, 44 age, permanent nahi kiya",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "sanitation_worker", "employer": "municipal_corporation", "employment_type": "contractual", "state": "Madhya Pradesh", "city": "Bhopal", "category": "SC", "age": 44, "work_years": 10}}, ensure_ascii=False)
    },
    {
        "input": "Mera beta army mein shaheed ho gaya, main 60 saal ki maa hoon, Maharashtra gaon mein, pension aati hai lekin aur kya mil sakta hai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"age": 60, "state": "Maharashtra", "area": "rural", "gender": "female", "family": [{"relation": "son", "status": "martyred_military"}], "has_pension": True, "looking_for": "additional_benefits"}}, ensure_ascii=False)
    },
    {
        "input": "Main driver hoon truck ka, long route, family Rajasthan mein, 38 age, 2 bachche, health insurance nahi hai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "truck_driver", "state": "Rajasthan", "age": 38, "gender": "male", "family": [{"relation": "children", "count": 2}], "has_insurance": False, "work_type": "long_route"}}, ensure_ascii=False)
    },
    {
        "input": "Sabzi mandi mein hammal ka kaam karta hoon, Azadpur Delhi, badan toot jaata hai, 50 age, UP ka hoon asal mein",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "porter_market", "work_state": "Delhi", "work_location": "Azadpur Mandi", "home_state": "Uttar Pradesh", "age": 50, "gender": "male", "migrant": True}}, ensure_ascii=False)
    },
    {
        "input": "I am a yoga teacher, run small classes in Rishikesh, want to register as MSME, 33 years, female",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "yoga_teacher", "state": "Uttarakhand", "city": "Rishikesh", "age": 33, "gender": "female", "business_type": "small_classes", "looking_for": "msme_registration"}}, ensure_ascii=False)
    },
    {
        "input": "Bidi worker hoon, Murshidabad se, mahila, 45 age, lungs ki problem ho gayi, koi health scheme hai kya",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "bidi_worker", "state": "West Bengal", "district": "Murshidabad", "age": 45, "gender": "female", "health_issue": "respiratory", "looking_for": "health_scheme"}}, ensure_ascii=False)
    },
    {
        "input": "NRI hoon, India wapas aa raha hoon permanently, age 45, Kerala, apna restaurant kholna hai",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "returning_nri", "state": "Kerala", "age": 45, "gender": "male", "aspiration": "restaurant_business"}}, ensure_ascii=False)
    },
    {
        "input": "Cooperative society mein doodh deta hoon, 4 bhains hain, Haryana, Jaat hoon, 38 age",
        "output": json.dumps({"intent": "scheme_discovery", "profile": {"occupation": "dairy_farmer", "state": "Haryana", "age": 38, "gender": "male", "livestock": {"buffalo": 4}, "sells_to": "cooperative"}}, ensure_ascii=False)
    },
]
print(f"Scheme extraction examples: {len(scheme_extraction_examples)}")

In [ ]:
import json
legal_extraction_examples = [
    {
        "input": "Mere malik ne 3 mahine se salary nahi di hai, private company mein kaam karta hoon Delhi mein",
        "output": json.dumps({"intent": "legal_aid", "category": "labor", "details": {"issue": "unpaid_wages", "duration_months": 3, "employer_type": "private_company", "state": "Delhi"}}, ensure_ascii=False)
    },
    {
        "input": "Mujhe bina notice nikaal diya company se, 5 saal kaam kiya, koi termination letter nahi diya, Noida",
        "output": json.dumps({"intent": "legal_aid", "category": "labor", "details": {"issue": "wrongful_termination", "years_worked": 5, "notice_given": False, "termination_letter": False, "state": "Uttar Pradesh", "city": "Noida"}}, ensure_ascii=False)
    },
    {
        "input": "Factory mein accident ho gaya, haath ki ungli kat gayi, malik compensation nahi de raha, Surat",
        "output": json.dumps({"intent": "legal_aid", "category": "labor", "details": {"issue": "workplace_injury", "injury": "finger_amputation", "compensation_denied": True, "employer_type": "factory", "state": "Gujarat", "city": "Surat"}}, ensure_ascii=False)
    },
    {
        "input": "PF ka paisa 2 saal se nahi aaya, company band ho gayi, EPF office bol raha file pending hai",
        "output": json.dumps({"intent": "legal_aid", "category": "labor", "details": {"issue": "pf_withdrawal_stuck", "pending_duration_years": 2, "company_status": "closed", "epf_response": "file_pending"}}, ensure_ascii=False)
    },
    {
        "input": "Boss office mein roz chillata hai, galiya deta hai, meri insult karta hai sabke saamne, IT company Bangalore",
        "output": json.dumps({"intent": "legal_aid", "category": "labor", "details": {"issue": "workplace_harassment", "harassment_type": "verbal_abuse", "frequency": "daily", "public": True, "employer_type": "IT_company", "state": "Karnataka", "city": "Bangalore"}}, ensure_ascii=False)
    },
    {
        "input": "Maternity leave nahi di company ne, bola resign karo, 8 mahine pregnant hoon, private firm Chennai",
        "output": json.dumps({"intent": "legal_aid", "category": "labor", "details": {"issue": "maternity_leave_denied", "pregnancy_months": 8, "employer_response": "asked_to_resign", "employer_type": "private_firm", "state": "Tamil Nadu", "city": "Chennai"}}, ensure_ascii=False)
    },
    {
        "input": "Maine online ek phone order kiya tha ₹15,000 ka, duplicate aaya hai, company return nahi le rahi",
        "output": json.dumps({"intent": "legal_aid", "category": "consumer", "details": {"issue": "counterfeit_product", "product": "phone", "amount": 15000, "purchase_mode": "online", "return_refused": True}}, ensure_ascii=False)
    },
    {
        "input": "Builder ne flat ka possession 3 saal late diya, registration bhi nahi hua, ₹40 lakh diye hain already",
        "output": json.dumps({"intent": "legal_aid", "category": "consumer", "details": {"issue": "delayed_possession", "property_type": "flat", "delay_years": 3, "registration_done": False, "amount_paid": 4000000, "against": "builder"}}, ensure_ascii=False)
    },
    {
        "input": "Hospital ne ₹2 lakh ka bill bana diya, operation zaroorat nahi tha doctor ne bola lekin kara diya",
        "output": json.dumps({"intent": "legal_aid", "category": "health", "details": {"issue": "unnecessary_surgery", "bill_amount": 200000, "against": "hospital", "negligence_type": "unnecessary_procedure"}}, ensure_ascii=False)
    },
    {
        "input": "Gym membership li thi 1 saal ki ₹25,000, 2 mahine baad gym band ho gaya, refund nahi de raha",
        "output": json.dumps({"intent": "legal_aid", "category": "consumer", "details": {"issue": "service_not_provided", "service": "gym_membership", "duration": "1_year", "used": "2_months", "amount": 25000, "refund_denied": True}}, ensure_ascii=False)
    },
    {
        "input": "Mera pati mujhe roz maarta hai, sasural wale bhi support nahi karte, mujhe kya karna chahiye",
        "output": json.dumps({"intent": "legal_aid", "category": "women", "details": {"issue": "domestic_violence", "violence_type": "physical", "frequency": "daily", "perpetrator": "husband", "family_support": False}, "emergency": True}, ensure_ascii=False)
    },
    {
        "input": "Sasural wale dahej maang rahe hain shaadi ke baad, 10 lakh aur chahiye, dhamki de rahe hain",
        "output": json.dumps({"intent": "legal_aid", "category": "women", "details": {"issue": "dowry_demand", "demand_amount": 1000000, "timing": "after_marriage", "threats": True, "perpetrator": "in_laws"}}, ensure_ascii=False)
    },
    {
        "input": "Office mein senior ne mujhe touch kiya inappropriately, baar baar messages bhejta hai, IT company Pune",
        "output": json.dumps({"intent": "legal_aid", "category": "women", "details": {"issue": "sexual_harassment_workplace", "harassment_type": ["physical_touch", "messages"], "perpetrator": "senior_colleague", "workplace_type": "IT_company", "state": "Maharashtra", "city": "Pune"}}, ensure_ascii=False)
    },
    {
        "input": "Pati ne talaq de diya aur ghar se nikaal diya, 2 bachche mere paas hain, koi income nahi hai",
        "output": json.dumps({"intent": "legal_aid", "category": "women", "details": {"issue": "divorce_maintenance", "divorced_by": "husband", "evicted": True, "children": 2, "children_with": "self", "income": "none"}}, ensure_ascii=False)
    },
    {
        "input": "Humari pushtaini zameen pe padosi ne kabza kar liya hai, koi kagaz bhi nahi dikhata",
        "output": json.dumps({"intent": "legal_aid", "category": "property", "details": {"issue": "encroachment", "property_type": "ancestral_land", "encroacher": "neighbor", "documents_shown": False}}, ensure_ascii=False)
    },
    {
        "input": "Kirayedar 2 saal se kiraya nahi de raha, ghar khaali bhi nahi karta, Mumbai",
        "output": json.dumps({"intent": "legal_aid", "category": "property", "details": {"issue": "tenant_not_paying", "unpaid_duration_years": 2, "eviction_refused": True, "state": "Maharashtra", "city": "Mumbai"}}, ensure_ascii=False)
    },
    {
        "input": "Papa ki death ke baad bhai sari property apne naam kar raha hai, mujhe hissa nahi de raha, main behen hoon",
        "output": json.dumps({"intent": "legal_aid", "category": "property", "details": {"issue": "inheritance_dispute", "property_after": "father_death", "denied_by": "brother", "claimant_gender": "female", "relation": "sister"}}, ensure_ascii=False)
    },
    {
        "input": "Mera UPI se ₹25,000 kisi ne nikal liye, fake link bhejke, kya karun",
        "output": json.dumps({"intent": "legal_aid", "category": "cyber", "details": {"issue": "upi_fraud", "method": "phishing_link", "amount_lost": 25000}}, ensure_ascii=False)
    },
    {
        "input": "Kisi ne meri fake profile banayi hai Instagram pe, mere photos use kar raha hai, morphed bhi",
        "output": json.dumps({"intent": "legal_aid", "category": "cyber", "details": {"issue": "fake_profile", "platform": "Instagram", "misuse": ["photos_stolen", "morphed_images"]}}, ensure_ascii=False)
    },
    {
        "input": "Online loan app wale roz phone karte hain, gaaliyan dete hain, contacts ko bhi message bheja, ₹5000 ka loan tha sirf",
        "output": json.dumps({"intent": "legal_aid", "category": "cyber", "details": {"issue": "loan_app_harassment", "harassment_type": ["calls", "abuse", "contact_list_misuse"], "loan_amount": 5000}}, ensure_ascii=False)
    },
    {
        "input": "Mujhe RTI file karni hai, humara area mein sadak 2 saal se nahi bani, MLA ne promise kiya tha",
        "output": json.dumps({"intent": "legal_aid", "category": "rti", "details": {"issue": "unfulfilled_promise", "regarding": "road_construction", "pending_years": 2, "promised_by": "MLA", "department": "PWD"}}, ensure_ascii=False)
    },
    {
        "input": "Humara ration card nahi ban raha 1 saal se, office wale baar baar chakkr lagwa rahe, RTI daalun kya",
        "output": json.dumps({"intent": "legal_aid", "category": "rti", "details": {"issue": "delayed_government_service", "service": "ration_card", "pending_years": 1, "department": "food_supply"}}, ensure_ascii=False)
    },
    {
        "input": "Park ki jagah builder ne mall bana diya, municipal corporation ne permission kaise di, jaanana hai",
        "output": json.dumps({"intent": "legal_aid", "category": "rti", "details": {"issue": "unauthorized_construction", "public_land": "park", "constructed": "mall", "question_to": "municipal_corporation", "seeking": "permission_details"}}, ensure_ascii=False)
    },
    {
        "input": "Mera phone chori ho gaya, police FIR nahi likh rahi, bol rahe itne mein kya hoga",
        "output": json.dumps({"intent": "legal_aid", "category": "police", "details": {"issue": "fir_not_registered", "crime": "phone_theft", "police_response": "refused"}}, ensure_ascii=False)
    },
    {
        "input": "Ghar pe dhamki di hai kisine, jaan se maarenge bola, neighbour ka ladka hai, Lucknow",
        "output": json.dumps({"intent": "legal_aid", "category": "police", "details": {"issue": "death_threat", "perpetrator": "neighbor_son", "location": "home", "state": "Uttar Pradesh", "city": "Lucknow"}, "emergency": True}, ensure_ascii=False)
    },
    {
        "input": "Papa retired hue 6 mahine ho gaye, pension abhi tak shuru nahi hui, government school teacher the, MP",
        "output": json.dumps({"intent": "legal_aid", "category": "pension", "details": {"issue": "pension_not_started", "retired_months_ago": 6, "previous_job": "government_school_teacher", "state": "Madhya Pradesh"}}, ensure_ascii=False)
    },
    {
        "input": "Widow pension band ho gayi 4 mahine se, pehle aati thi ₹500, ab koi response nahi, Odisha",
        "output": json.dumps({"intent": "legal_aid", "category": "pension", "details": {"issue": "pension_stopped", "pension_type": "widow_pension", "amount": 500, "stopped_months": 4, "state": "Odisha"}}, ensure_ascii=False)
    },
    {
        "input": "School ne admission nahi diya RTE quota mein, bol rahe seat nahi hai, private school hai Jaipur ka",
        "output": json.dumps({"intent": "legal_aid", "category": "education", "details": {"issue": "rte_admission_denied", "school_type": "private", "reason_given": "no_seats", "state": "Rajasthan", "city": "Jaipur"}}, ensure_ascii=False)
    },
    {
        "input": "College ne TC rok li hai fees ke liye, ₹12,000 baaki hai, exam nahi dene de rahe",
        "output": json.dumps({"intent": "legal_aid", "category": "education", "details": {"issue": "tc_withheld", "reason": "pending_fees", "amount_due": 12000, "exam_blocked": True}}, ensure_ascii=False)
    },
    {
        "input": "Doctor ne galat operation kar diya, left kidney nikalni thi right nikal di, Bhopal hospital",
        "output": json.dumps({"intent": "legal_aid", "category": "health", "details": {"issue": "medical_negligence", "negligence_type": "wrong_side_surgery", "what_happened": "removed_right_kidney_instead_of_left", "state": "Madhya Pradesh", "city": "Bhopal"}}, ensure_ascii=False)
    },
    {
        "input": "Bacche ko vaccine lagayi private clinic ne, reaction aa gaya, 3 din ICU mein raha, ₹1.5 lakh ka bill",
        "output": json.dumps({"intent": "legal_aid", "category": "health", "details": {"issue": "vaccine_adverse_reaction", "patient": "child", "clinic_type": "private", "outcome": "ICU_3_days", "bill_amount": 150000}}, ensure_ascii=False)
    },
    {
        "input": "Padosi raat 12 baje tak DJ bajata hai, police kuch nahi karti, roz ka hai yeh, Gurgaon",
        "output": json.dumps({"intent": "legal_aid", "category": "police", "details": {"issue": "noise_pollution", "source": "neighbor_DJ", "time": "past_midnight", "frequency": "daily", "police_response": "no_action", "state": "Haryana", "city": "Gurgaon"}}, ensure_ascii=False)
    },
    {
        "input": "Car ka insurance claim reject kar diya, accident hua tha genuine, ₹80,000 ka repair, company bol rahi policy lapse thi but maine renew kiya tha",
        "output": json.dumps({"intent": "legal_aid", "category": "consumer", "details": {"issue": "insurance_claim_rejected", "insurance_type": "car", "reason_given": "policy_lapse", "user_claim": "was_renewed", "claim_amount": 80000}}, ensure_ascii=False)
    },
    {
        "input": "Health insurance wale papa ka heart surgery claim reject kar rahe, bol rahe pre-existing condition, policy 3 saal purani hai",
        "output": json.dumps({"intent": "legal_aid", "category": "consumer", "details": {"issue": "health_insurance_rejected", "procedure": "heart_surgery", "reason_given": "pre_existing_condition", "policy_age_years": 3, "patient": "father"}}, ensure_ascii=False)
    },
    {
        "input": "Meri problem hai kisi ne mera paisa loot liya",
        "output": json.dumps({"intent": "legal_aid", "category": "unknown", "details": {"issue": "money_stolen"}, "missing_info": ["how_stolen", "amount", "who_did_it", "location", "online_or_physical"], "follow_up_hindi": "Paisa kaise gaya? Online fraud tha ya kisi ne physically loota? Kitna amount tha?"}, ensure_ascii=False)
    },
    {
        "input": "help karo yaar bahut pareshan hoon",
        "output": json.dumps({"intent": "general_query", "details": {}, "missing_info": ["what_problem", "category"], "follow_up_hindi": "Kya problem hai? Scheme dhundh rahe ho ya koi legal issue hai? Detail mein bataiye."}, ensure_ascii=False)
    },
]
print(f"Legal extraction examples: {len(legal_extraction_examples)}")

In [ ]:
import json
document_extraction_examples = [
    {
        "input": "[Document image: Aadhaar card. Name: Ramesh Kumar. DOB: 15/03/1978. Gender: Male. Address: Village Rampur, Block Sadar, District Jaipur, Rajasthan 302001. Aadhaar: 8765 4321 9012]",
        "output": json.dumps({"intent": "document_scan", "document_type": "aadhaar", "extracted": {"name": "Ramesh Kumar", "date_of_birth": "15/03/1978", "gender": "Male", "address": "Village Rampur, Block Sadar, District Jaipur, Rajasthan 302001", "state": "Rajasthan", "aadhaar_number": "8765-4321-9012"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Aadhaar card. Name: Sunita Devi. DOB: 22/08/1990. Gender: Female. Address: Mohalla Nai Basti, Ward 5, Allahabad, UP 211001. Aadhaar: 1234 5678 9012]",
        "output": json.dumps({"intent": "document_scan", "document_type": "aadhaar", "extracted": {"name": "Sunita Devi", "date_of_birth": "22/08/1990", "gender": "Female", "address": "Mohalla Nai Basti, Ward 5, Allahabad, UP 211001", "state": "Uttar Pradesh", "aadhaar_number": "1234-5678-9012"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Aadhaar card. Name in Hindi: मोहम्मद इस्माइल. DOB: 01/01/1965. Gender: Male. Address: 45 Zakir Nagar, Okhla, New Delhi 110025]",
        "output": json.dumps({"intent": "document_scan", "document_type": "aadhaar", "extracted": {"name": "Mohammed Ismail", "name_hindi": "मोहम्मद इस्माइल", "date_of_birth": "01/01/1965", "gender": "Male", "address": "45 Zakir Nagar, Okhla, New Delhi 110025", "state": "Delhi"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: PAN Card. Name: SUNITA DEVI. Father's Name: RAM PRASAD. DOB: 22/08/1985. PAN: ABCDE1234F]",
        "output": json.dumps({"intent": "document_scan", "document_type": "pan_card", "extracted": {"name": "SUNITA DEVI", "father_name": "RAM PRASAD", "date_of_birth": "22/08/1985", "pan_number": "ABCDE1234F"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: PAN Card. Name: VIKASH SHARMA. Father: RAJENDRA SHARMA. DOB: 14/11/1992. PAN: BZSPS5678K]",
        "output": json.dumps({"intent": "document_scan", "document_type": "pan_card", "extracted": {"name": "VIKASH SHARMA", "father_name": "RAJENDRA SHARMA", "date_of_birth": "14/11/1992", "pan_number": "BZSPS5678K"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Ration Card BPL. Head of Family: Mohammed Ismail. Members: 5. Category: BPL. State: Tamil Nadu. Card No: TN-BPL-123456]",
        "output": json.dumps({"intent": "document_scan", "document_type": "ration_card", "extracted": {"head_of_family": "Mohammed Ismail", "total_members": 5, "category": "BPL", "state": "Tamil Nadu", "card_number": "TN-BPL-123456"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Ration Card APL. Head: Geeta Bai. Members: 3. Category: APL. State: Rajasthan. Card: RJ-APL-789012]",
        "output": json.dumps({"intent": "document_scan", "document_type": "ration_card", "extracted": {"head_of_family": "Geeta Bai", "total_members": 3, "category": "APL", "state": "Rajasthan", "card_number": "RJ-APL-789012"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Voter ID (EPIC). Name: Rajesh Kumar Yadav. Father: Shivnath Yadav. DOB: 05/06/1982. Address: Gram Pipra, Block Sahebganj, Muzaffarpur, Bihar. EPIC No: VDL1234567]",
        "output": json.dumps({"intent": "document_scan", "document_type": "voter_id", "extracted": {"name": "Rajesh Kumar Yadav", "father_name": "Shivnath Yadav", "date_of_birth": "05/06/1982", "address": "Gram Pipra, Block Sahebganj, Muzaffarpur, Bihar", "state": "Bihar", "epic_number": "VDL1234567"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Caste Certificate. Name: Amit Kumar. Father: Shyam Lal. Caste: Chamar. Category: Scheduled Caste. State: Uttar Pradesh. Issued by: District Magistrate, Agra. Date: 12/04/2020]",
        "output": json.dumps({"intent": "document_scan", "document_type": "caste_certificate", "extracted": {"name": "Amit Kumar", "father_name": "Shyam Lal", "caste": "Chamar", "category": "SC", "state": "Uttar Pradesh", "issued_by": "District Magistrate, Agra", "issue_date": "12/04/2020"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Income Certificate. Name: Priya Kumari. Annual Income: Rs 72,000. Issued by: Tehsildar, Patna. Valid till: March 2025. Family members: 4]",
        "output": json.dumps({"intent": "document_scan", "document_type": "income_certificate", "extracted": {"name": "Priya Kumari", "annual_income": 72000, "issued_by": "Tehsildar, Patna", "valid_till": "March 2025", "family_members": 4, "state": "Bihar"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Salary slip. Employee: Ankit Verma. Company: ABC Pvt Ltd. Month: March 2025. Basic: 18000. HRA: 9000. DA: 3600. Gross: 30600. PF deduction: 2160. Net: 28440]",
        "output": json.dumps({"intent": "document_scan", "document_type": "salary_slip", "extracted": {"employee_name": "Ankit Verma", "company": "ABC Pvt Ltd", "month": "March 2025", "basic": 18000, "hra": 9000, "da": 3600, "gross": 30600, "pf_deduction": 2160, "net_salary": 28440}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Khatauni (land record). Owner: Harpal Singh. Village: Khanpur. Tehsil: Saharanpur. Khasra No: 234/1. Area: 2.5 bigha. Crop: Sugarcane. State: UP]",
        "output": json.dumps({"intent": "document_scan", "document_type": "land_record", "extracted": {"owner": "Harpal Singh", "village": "Khanpur", "tehsil": "Saharanpur", "khasra_number": "234/1", "area": "2.5 bigha", "current_crop": "Sugarcane", "state": "Uttar Pradesh"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Government letter in Hindi. From: District Collector Jhansi. To: Smt Kamla Devi. Subject: PM Awas Yojana application status. Content: Application PMAY-UP-2024-78901 under review, field verification pending]",
        "output": json.dumps({"intent": "document_scan", "document_type": "government_letter", "extracted": {"from": "District Collector, Jhansi", "to": "Smt Kamla Devi", "regarding": "PM Awas Yojana application", "application_no": "PMAY-UP-2024-78901", "status": "under_review", "pending_step": "field_verification"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Bank Passbook first page. Name: Ramesh Prasad. Account: 3201XXXXXXX890. Bank: State Bank of India. Branch: Varanasi Main. IFSC: SBIN0001234. Nominee: Savitri Devi]",
        "output": json.dumps({"intent": "document_scan", "document_type": "bank_passbook", "extracted": {"account_holder": "Ramesh Prasad", "account_number": "3201XXXXXXX890", "bank": "State Bank of India", "branch": "Varanasi Main", "ifsc": "SBIN0001234", "nominee": "Savitri Devi"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Electricity bill. Consumer: Rajni Sharma. Connection No: 1234567890. Address: B-45, Sector 21, Noida. Units consumed: 320. Bill amount: Rs 2,450. Due date: 15/06/2025]",
        "output": json.dumps({"intent": "document_scan", "document_type": "electricity_bill", "extracted": {"consumer_name": "Rajni Sharma", "connection_number": "1234567890", "address": "B-45, Sector 21, Noida", "units": 320, "bill_amount": 2450, "due_date": "15/06/2025", "state": "Uttar Pradesh"}}, ensure_ascii=False)
    },
    {
        "input": "[Document image: Disability Certificate. Name: Suresh Babu. Disability: Locomotor (both legs). Percentage: 75%. Permanent. Issued by: Civil Hospital Bhopal. UDID: MP-DIS-2023-45678]",
        "output": json.dumps({"intent": "document_scan", "document_type": "disability_certificate", "extracted": {"name": "Suresh Babu", "disability_type": "locomotor", "affected": "both_legs", "percentage": 75, "permanent": True, "issued_by": "Civil Hospital Bhopal", "udid_number": "MP-DIS-2023-45678", "state": "Madhya Pradesh"}}, ensure_ascii=False)
    },
]
print(f"Document extraction examples: {len(document_extraction_examples)}")

In [ ]:
from datasets import Dataset
_need = ("scheme_extraction_examples", "legal_extraction_examples", "document_extraction_examples", "SYSTEM_PROMPT")
_bad = [n for n in _need if n not in globals()]
if _bad:
    raise RuntimeError("Run Cells 3, 4, and 5 (data) before this cell. Missing: " + ", ".join(_bad))
all_examples = scheme_extraction_examples + legal_extraction_examples + document_extraction_examples
training_data = []
for ex in all_examples:
    training_data.append({
        "text": f"""<start_of_turn>user
{SYSTEM_PROMPT}
User says: {ex['input']}<end_of_turn>
<start_of_turn>model
{ex['output']}<end_of_turn>"""
    })
print(f"Total training examples: {len(training_data)}")
print(f"  - Scheme profile extraction: {len(scheme_extraction_examples)}")
print(f"  - Legal situation extraction: {len(legal_extraction_examples)}")
print(f"  - Document field extraction:  {len(document_extraction_examples)}")
dataset = Dataset.from_list(training_data)
print(f"\nDataset ready: {len(dataset)} examples")
print(f"\nSample:")
print(dataset[0]['text'][:500])

In [ ]:
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
for _name in ("model", "tokenizer", "dataset", "MAX_SEQ_LENGTH"):
    if _name not in globals():
        raise RuntimeError(f"Missing `{_name}` — run Cell 2 and Cells 3→6 before training (Cell 7).")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        max_steps=200,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="sahaj_outputs",
        optim="adamw_8bit",
        seed=42,
    ),
)
print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
import gc
import os
import pathlib
import shutil
import subprocess
import sys
import torch
from huggingface_hub import save_torch_state_dict
from peft import get_peft_model_state_dict
from unsloth import FastLanguageModel
if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("Run Cell 2 and Cell 7 first (same session).")
WORKDIR = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").is_dir() else pathlib.Path(".")
LORA_DIR = WORKDIR / "sahaj_lora"
MERGED_DIR = WORKDIR / "sahaj_merged"
LLAMA_CPP = WORKDIR / "llama.cpp"
GGUF_FINAL = WORKDIR / "sahaj_gemma4_q4_k_m.gguf"
F16_TMP = WORKDIR / "_sahaj_f16_tmp.gguf"
SKIP_MERGE = False
if not SKIP_MERGE and (WORKDIR / "sahaj_gguf" / "config.json").is_file():
    MERGED_DIR = WORKDIR / "sahaj_gguf"
    SKIP_MERGE = True
    print("Found existing merge at sahaj_gguf/ — SKIP_MERGE=True")
def _disk_gb():
    st = shutil.disk_usage(WORKDIR)
    return st.used / (1024**3), st.total / (1024**3)
def _cleanup_before_gguf(keep_merged=True):
    for p in (WORKDIR / "sahaj_outputs", WORKDIR / "unsloth_compiled_cache"):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
            print("Removed", p.name)
    if not keep_merged and MERGED_DIR.is_dir() and MERGED_DIR != WORKDIR / "sahaj_gguf":
        shutil.rmtree(MERGED_DIR, ignore_errors=True)
    for z in (WORKDIR / "sahaj_gguf.zip", F16_TMP):
        if z.is_file():
            z.unlink()
def _clone_llama_cpp():
    if (LLAMA_CPP / "convert_hf_to_gguf.py").is_file():
        return
    print("Cloning llama.cpp...")
    if LLAMA_CPP.exists():
        shutil.rmtree(LLAMA_CPP, ignore_errors=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP)],
        check=True,
        timeout=600,
    )
def _find_quantize_bin():
    candidates = [
        LLAMA_CPP / "build" / "bin" / "llama-quantize",
        LLAMA_CPP / "llama-quantize",
        pathlib.Path("/root/.unsloth/llama.cpp/build/bin/llama-quantize"),
        pathlib.Path("/root/.unsloth/llama.cpp/llama-quantize"),
    ]
    for p in candidates:
        if p.is_file():
            return p
    print("Building llama-quantize (2-4 min)...")
    subprocess.run(
        ["cmake", "-B", "build", "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_CURL=OFF"],
        cwd=str(LLAMA_CPP),
        check=True,
    )
    subprocess.run(
        ["cmake", "--build", "build", "-j2", "--target", "llama-quantize"],
        cwd=str(LLAMA_CPP),
        check=True,
    )
    return LLAMA_CPP / "build" / "bin" / "llama-quantize"
def _hf_to_gguf_q4(merged_dir: pathlib.Path):
    _clone_llama_cpp()
    if F16_TMP.is_file():
        F16_TMP.unlink()
    if GGUF_FINAL.is_file():
        GGUF_FINAL.unlink()
    print("HF → GGUF f16...")
    r = subprocess.run(
        [sys.executable, "convert_hf_to_gguf.py", str(merged_dir), "--outfile", str(F16_TMP), "--outtype", "f16"],
        cwd=str(LLAMA_CPP),
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print(r.stdout[-2000:] if r.stdout else "")
        print(r.stderr[-2000:] if r.stderr else "")
        raise RuntimeError("convert_hf_to_gguf.py failed (see log above)")
    for f in merged_dir.glob("model*.safetensors"):
        print("Deleting shard:", f.name)
        f.unlink()
    used, total = _disk_gb()
    print(f"Disk after f16: {used:.1f} / {total:.1f} GiB")
    qbin = _find_quantize_bin()
    print("Quantizing → Q4_K_M...")
    subprocess.run([str(qbin), str(F16_TMP), str(GGUF_FINAL), "Q4_K_M"], check=True)
    F16_TMP.unlink(missing_ok=True)
    print(f"GGUF ready: {GGUF_FINAL} ({GGUF_FINAL.stat().st_size / (1024**3):.2f} GB)")
LORA_DIR.mkdir(parents=True, exist_ok=True)
print("Saving LoRA...")
lora_sd = get_peft_model_state_dict(model)
save_torch_state_dict(
    lora_sd,
    str(LORA_DIR),
    max_shard_size="2GB",
    safe_serialization=True,
    filename_pattern="adapter_model{suffix}.safetensors",
)
if hasattr(model, "peft_config") and model.peft_config:
    for cfg in model.peft_config.values():
        cfg.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))
print("LoRA saved:", LORA_DIR)
for _nm in ("trainer", "trainer_stats"):
    if _nm in globals():
        del globals()[_nm]
gc.collect()
torch.cuda.empty_cache()
_cleanup_before_gguf(keep_merged=SKIP_MERGE)
if not SKIP_MERGE:
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    print("Merging LoRA → 16-bit HF...")
    FastLanguageModel.for_inference(model)
    model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
    print("Merged:", MERGED_DIR)
else:
    print("Using existing merged weights:", MERGED_DIR)
if not (MERGED_DIR / "config.json").is_file():
    raise FileNotFoundError(f"No config.json in {MERGED_DIR} — merge failed or wrong path")
used, total = _disk_gb()
print(f"Disk before GGUF convert: {used:.1f} / {total:.1f} GiB")
_hf_to_gguf_q4(MERGED_DIR)
os.chdir(WORKDIR)
subprocess.run(["zip", "-j", "sahaj_gguf.zip", str(GGUF_FINAL)], check=False)
subprocess.run(["zip", "-r", "sahaj_lora.zip", "sahaj_lora"], check=False)
print("Download from Output: sahaj_gguf.zip + sahaj_lora.zip")


In [ ]:
import pathlib, shutil, subprocess, sys
WORKDIR = pathlib.Path("/kaggle/working")
MERGED_DIR = WORKDIR / "sahaj_gguf"
LLAMA_CPP = WORKDIR / "llama.cpp"
GGUF_FINAL = WORKDIR / "sahaj_gemma4_q4_k_m.gguf"
F16_TMP = WORKDIR / "_sahaj_f16_tmp.gguf"
assert (MERGED_DIR / "config.json").is_file(), "Run merge first or use full Cell 8"
for p in (WORKDIR / "sahaj_outputs", WORKDIR / "unsloth_compiled_cache"):
    shutil.rmtree(p, ignore_errors=True)
if not (LLAMA_CPP / "convert_hf_to_gguf.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP)], check=True)
subprocess.run(
    [sys.executable, "convert_hf_to_gguf.py", str(MERGED_DIR), "--outfile", str(F16_TMP), "--outtype", "f16"],
    cwd=str(LLAMA_CPP),
    check=True,
)
for f in MERGED_DIR.glob("model*.safetensors"):
    f.unlink()
qbin = LLAMA_CPP / "build" / "bin" / "llama-quantize"
if not qbin.is_file():
    qbin = pathlib.Path("/root/.unsloth/llama.cpp/build/bin/llama-quantize")
if not qbin.is_file():
    subprocess.run(["cmake", "-B", "build", "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_BUILD_TESTS=OFF"], cwd=str(LLAMA_CPP), check=True)
    subprocess.run(["cmake", "--build", "build", "-j2", "--target", "llama-quantize"], cwd=str(LLAMA_CPP), check=True)
    qbin = LLAMA_CPP / "build" / "bin" / "llama-quantize"
subprocess.run([str(qbin), str(F16_TMP), str(GGUF_FINAL), "Q4_K_M"], check=True)
F16_TMP.unlink(missing_ok=True)
subprocess.run(["zip", "-j", "sahaj_gguf.zip", str(GGUF_FINAL)], cwd=str(WORKDIR), check=True)
print("Done:", GGUF_FINAL)

In [ ]:
import json
import torch
if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("Run Cell 2 (model setup) and training (Cell 7) before this test cell.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this QLoRA workflow (use a Kaggle GPU session).")
model.eval()
if "SYSTEM_PROMPT" not in globals():
    SYSTEM_PROMPT = """You are Sahaj, an AI assistant for Indian citizens.
Your job: understand the user's input and extract structured information.
Always respond with valid JSON containing:
- "intent": one of ["scheme_discovery", "legal_aid", "document_scan", "general_query"]
- relevant extracted fields based on intent
Do NOT suggest specific schemes or laws. Only extract the user's profile/situation."""
test_inputs = [
    "Main taxi driver hoon Jaipur mein, 42 age, 2 bache padh rahe hain, loan chahiye gaadi ke liye",
    "Meri maa blind hain, 70 saal, Assam gaon, koi pension ya help mil sakti hai kya",
    "I'm a 22 year old girl from Kerala, just finished B.Tech, want to start a startup",
    "Landlord ne bina notice ghar khaali karne bola, 5 saal se rehte hain, agreement hai",
    "Amazon se order kiya ₹8000 ka laptop bag, empty box aaya, refund nahi mil raha",
    "Kisi ne mere naam pe loan le liya, mujhe bank ka notice aaya, identity theft hua hai",
    "bhai kuch kaam nahi mil raha",
]
print("=" * 70)
print("SAHAJ - EXTRACTION TEST (unseen inputs)")
print("=" * 70)
for i, test_input in enumerate(test_inputs, 1):
    prompt = f"""<start_of_turn>user
{SYSTEM_PROMPT}
User says: {test_input}<end_of_turn>
<start_of_turn>model
"""
    device = torch.device("cuda")
    inputs = tokenizer(text=prompt, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    gen_ids = outputs[0][input_len:]
    model_output = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    print(f"\n--- Test {i} ---")
    print(f"INPUT:  {test_input}")
    try:
        parsed = json.loads(model_output)
        print(f"INTENT: {parsed.get('intent')}")
        if 'profile' in parsed:
            print(f"PROFILE: {json.dumps(parsed['profile'], indent=2, ensure_ascii=False)}")
        if 'details' in parsed:
            print(f"DETAILS: {json.dumps(parsed['details'], indent=2, ensure_ascii=False)}")
        if 'missing_info' in parsed:
            print(f"MISSING: {parsed['missing_info']}")
            print(f"FOLLOW-UP: {parsed.get('follow_up_hindi', '')}")
        print("JSON VALID: Yes")
    except json.JSONDecodeError:
        print(f"RAW OUTPUT: {model_output[:300]}")
        print("JSON VALID: No")
    print("-" * 70)

In [ ]:
import os
import pathlib
import shutil
WORKDIR = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").is_dir() else pathlib.Path(".")
output_dir = WORKDIR / "sahaj_submission"
lora_src = WORKDIR / "sahaj_lora"
gguf_src = WORKDIR / "sahaj_gguf"
if gguf_src.is_dir() and not any(gguf_src.rglob("*.gguf")):
    print("Removing partial sahaj_gguf/ (no .gguf) to free disk...")
    shutil.rmtree(gguf_src, ignore_errors=True)
if output_dir.exists():
    shutil.rmtree(output_dir, ignore_errors=True)
output_dir.mkdir(parents=True, exist_ok=True)
if lora_src.is_dir():
    shutil.copytree(lora_src, output_dir / "sahaj_lora", dirs_exist_ok=True)
    print("Copied sahaj_lora/")
elif (WORKDIR / "sahaj_lora.zip").is_file():
    print("LoRA folder missing — use sahaj_lora.zip from Output (Cell 8).")
else:
    print("Warning: run Cell 8 first.")
if gguf_src.is_dir() and any(gguf_src.rglob("*.gguf")):
    shutil.copytree(gguf_src, output_dir / "sahaj_gguf", dirs_exist_ok=True)
    print("Copied sahaj_gguf/ (.gguf present)")
else:
    print("(No GGUF — LoRA-only submission is OK for hackathon.)")
print("Submission folder:", output_dir)
for fp in sorted(output_dir.rglob("*")):
    if fp.is_file():
        print(f"  {fp} ({fp.stat().st_size / 1024 / 1024:.1f} MB)")